# PDZ and CRIPT plots by proposed paper section

This notebook organizes the current PDZ and CRIPT analyses around the first three proposed paper sections: reported-order recovery, posterior rank uncertainty, and top-$k$ decision stability. All eight completed analyses are retained in each applicable landscape-specific block.

Two score definitions are shown separately throughout. `log_mean_growth` is the posterior latent assay score. `count_ratio` is computed from posterior-predictive input and output counts, so it includes the variability of a replicated sequencing experiment and answers a different question.

The figures read saved `metadata.json`, `summary.csv`, `overlaps.csv`, and, through the plotting functions used here, their plot-ready summaries. They do not reopen an HMC fit. Statistical concerns and proposed checks for the paper interpretation are recorded in [PAPER_CONCERNS_AND_PLAN.md](PAPER_CONCERNS_AND_PLAN.md).

## Configuration

The eight result directories live under the same PDZ result root, with alternate fits nested by model beneath their library directory. The checks below require analysis settings that should agree across runs while allowing the model and posterior sample count to differ.

In [ ]:
# ruff: noqa: E402
# pyright: reportMissingImports=false

import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from matplotlib.figure import Figure


def find_project_root() -> Path:
    for base in (Path.cwd(), *Path.cwd().parents):
        for candidate in (base, base / "dms-fitness-stability"):
            if (candidate / "src" / "dms_fitness_stability").is_dir():
                return candidate
    raise FileNotFoundError("Could not find the dms-fitness-stability repository.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from dms_fitness_stability import (
    plot_calibration_divergence,
    plot_calibration_overlay,
    plot_rank_divergence_boxplots,
    plot_report,
)
from dms_fitness_stability.adapters.pdz import PDZ_LANDSCAPES

DATA_ROOT = Path(os.environ.get("DMS_FITNESS_STABILITY_DATA", "/data/data_dms_stability"))
RESULT_ROOT = DATA_ROOT / "pdz" / "results"
SCORES = ("log_mean_growth", "count_ratio")
SCORE_TITLES = {
    "log_mean_growth": "Log mean growth",
    "count_ratio": "Count ratio",
}
K = 50
INDIVIDUAL_REPORTED_TOP_N = 50
INCLUSION_DISPLAY_N = 50
RANK_DIVERGENCE_COHORTS = (100, 1000)

RESULT_DIRS = (
    RESULT_ROOT / "trans-1",
    RESULT_ROOT / "trans-2",
    RESULT_ROOT / "cis",
    RESULT_ROOT / "cis" / "exponential-sigmoid",
    RESULT_ROOT / "cis" / "lomax-exponential",
    RESULT_ROOT / "cript-c" / "exponential-exponential",
    RESULT_ROOT / "cript-c" / "exponential-sigmoid",
    RESULT_ROOT / "cript-n" / "gamma-sigmoid",
)
metadata = {}
RESULTS = []
for path in RESULT_DIRS:
    row = json.loads((path / "metadata.json").read_text(encoding="utf-8"))
    label = f"{row['source']} / {row['model']}"
    if label in metadata:
        raise ValueError(f"Duplicate PDZ analysis label: {label}")
    metadata[label] = row
    RESULTS.append((label, path))

found_landscapes = {row["source"] for row in metadata.values()}
if found_landscapes != set(PDZ_LANDSCAPES):
    raise ValueError(
        f"Expected PDZ landscapes {PDZ_LANDSCAPES}, got {sorted(found_landscapes)}"
    )

shared_fields = (
    "schema_version",
    "fit_method",
    "scores",
    "retained_k",
    "tracked_reported_top_n",
    "top_k",
    "interval",
    "seed",
    "reference_fitness_source",
    "reported_fitness_available",
    "posterior_predictive_count_source",
    "observed_counts_status",
)
for field in shared_fields:
    values = {json.dumps(row[field], sort_keys=True) for row in metadata.values()}
    if len(values) != 1:
        raise ValueError(f"PDZ analyses disagree on {field}: {values}")

for label, row in metadata.items():
    missing_scores = set(SCORES) - set(row["scores"])
    if missing_scores:
        raise ValueError(f"{label} is missing requested scores: {sorted(missing_scores)}")


def display_result_report(
    label: str,
    result_dir: Path,
    *,
    views: tuple[str, ...],
    n_variants: int = 40,
) -> dict[str, Figure]:
    display(Markdown(f"#### {label}"))
    figures = plot_report(
        result_dir,
        k=K,
        scores=SCORES,
        views=views,
        n_variants=n_variants,
        reported_top_n=INDIVIDUAL_REPORTED_TOP_N,
    )
    for figure in figures.values():
        display(figure)
        plt.close(figure)
    return figures


print(f"Result root: {RESULT_ROOT}")
for label, _ in RESULTS:
    row = metadata[label]
    print(
        f"{label}: {row['n_variants']:,} reported candidates; "
        f"{row['n_hmc_variants']:,} fitted variants; "
        f"{row['n_samples']:,} posterior samples"
    )

## 1. Do we recover the reported ordering?

### Landscape-specific reported-rank posterior percentiles

For each analysis and score, the left panel uses the tracked reported top 1,000 and the right panel uses the reported top 50. These plots locate each published rank within its marginal posterior rank distribution.

The uniform diagonal is retained as a descriptive reference, but its interpretation as a null for agreement is provisional, especially after selecting variants by reported rank. See [PAPER_CONCERNS_AND_PLAN.md](PAPER_CONCERNS_AND_PLAN.md) before using the ECDF-to-diagonal area as a paper-level agreement statistic.

In [ ]:
rank_percentile_reports = {
    label: display_result_report(
        label,
        result_dir,
        views=("calibration",),
    )
    for label, result_dir in RESULTS
}

### Reported-rank posterior percentiles across analyses

These overlays compare the tracked reported top 1,000 across all eight landscape/model analyses. Each ECDF-to-uniform area is reported as a descriptive curve summary, not yet as a validated measure of ordering agreement. Both the latent assay score and posterior-predictive count ratio are kept visible because their difference is scientifically informative.

In [ ]:
rank_percentile_overlay_figures = {}
rank_percentile_tables = {}
ecdf_area_figures = {}
ecdf_area_tables = {}

for score in SCORES:
    display(Markdown(f"#### {SCORE_TITLES[score]}"))

    figure, _, percentiles = plot_calibration_overlay(
        RESULTS,
        score=score,
        k=K,
        cohort="tracked",
        alpha=0.8,
    )
    rank_percentile_overlay_figures[score] = figure
    rank_percentile_tables[score] = percentiles
    display(figure)
    plt.close(figure)

    figure, _, ecdf_areas = plot_calibration_divergence(
        RESULTS,
        score=score,
        k=K,
        cohort="tracked",
    )
    ecdf_area_figures[score] = figure
    ecdf_area_tables[score] = ecdf_areas
    display(figure)
    plt.close(figure)

    display(ecdf_areas.sort_values("calibration_l1_area").reset_index(drop=True))

## 2. How uncertain are the reported ranks?

### Landscape-specific posterior rank and score intervals

For every analysis and score, the rank plot shows the reported top 50 against its full-candidate posterior rank distribution. The green band marks the top-50 decision region. Posterior score intervals are retained as supporting plots even though the proposed paper section centers ranks rather than score magnitudes.

In [ ]:
rank_uncertainty_reports = {
    label: display_result_report(
        label,
        result_dir,
        views=("rank", "value"),
    )
    for label, result_dir in RESULTS
}

### Signed posterior-rank divergence across analyses

For each score, the boxplots summarize $Q_{0.05}(R_i)-r_i^{\mathrm{rep}}$ and $Q_{0.95}(R_i)-r_i^{\mathrm{rep}}$ across variants. The reported top 100 matches the proposed main-paper cohort; the reported top 1,000 is retained as a broader supporting view. Outliers are currently omitted from the rendered boxes.

In [ ]:
rank_divergence_figures = {score: {} for score in SCORES}
rank_divergence_tables = {score: {} for score in SCORES}

for score in SCORES:
    display(Markdown(f"#### {SCORE_TITLES[score]}"))

    for reported_top_n in RANK_DIVERGENCE_COHORTS:
        figure, _, divergence = plot_rank_divergence_boxplots(
            RESULTS,
            score=score,
            k=K,
            reported_top_n=reported_top_n,
        )
        rank_divergence_figures[score][reported_top_n] = figure
        rank_divergence_tables[score][reported_top_n] = divergence
        display(figure)
        plt.close(figure)

## 3. How stable are top-$k$ decisions?

### Landscape-specific top-50 inclusion and overlap

For every analysis and score, the inclusion plot shows the 50 variants with the largest posterior probability of entering the top 50. It marks whether each variant belongs to the reported top 50, so the display can contain both stable reported candidates and outside challengers. This is useful for the proposed inclusion-probability and candidate-cloud panels, but it is not restricted to all and only the reported top-50 variants.

The overlap plot shows two distributions at $k=50$: posterior draw versus posterior draw, and posterior draw versus the reported top-50 set. The latter directly summarizes the fixed-$k$ overlap in the paper outline. A full $O_k$ curve is not plotted here yet, although the saved draw-level top-1,000 selections are sufficient to construct it later.

In [ ]:
top_k_stability_reports = {
    label: display_result_report(
        label,
        result_dir,
        views=("inclusion", "overlap"),
        n_variants=INCLUSION_DISPLAY_N,
    )
    for label, result_dir in RESULTS
}